In [1]:
library(tidyverse)
library(BiocParallel)
library(parallel)
library(sva)

# Custom package
library(rutils)

Warning message in system("timedatectl", intern = TRUE):
“running command 'timedatectl' had status 1”
── Attaching packages ─────────────────────────────────────────────────────────────────────────────── tidyverse 1.3.1 ──

✔ ggplot2 3.3.5     ✔ purrr   0.3.4
✔ tibble  3.1.2     ✔ dplyr   1.0.7
✔ tidyr   1.1.3     ✔ stringr 1.4.0
✔ readr   2.0.0     ✔ forcats 0.5.1

── Conflicts ────────────────────────────────────────────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()

Loading required package: mgcv

Loading required package: nlme


Attaching package: ‘nlme’


The following object is masked from ‘package:dplyr’:

    collapse


This is mgcv 1.8-36. For overview type 'help("mgcv-package")'.

Loading required package: genefilter


Attaching package: ‘genefilter’


The following object is masked from ‘package:readr’:

    spec




In [2]:
n_cores <- detectCores() - 2
# Might be different depening on OS
mc_param <- SnowParam(n_cores)

# Load data

In [3]:
dsets <- c("GSE4888", "GSE6364", "GSE7305", "GSE51981")
batches <- as.list(setNames(seq(1, length(dsets)), dsets))

dirs <- rutils::get_dev_directories("../dev_paths.txt")
lnames <- load("../proj/proj_vals.Rdata")

In [4]:
matrisome_df <- rutils::load_matrisome_df(paste0(dirs$data_dir, "/matrisome/matrisome_hs_masterlist.tsv")) %>%
    select(gene_symbol, division, category)
rma_dfs <- list()
clinical_dfs <- list()

for (ds in dsets) {
    rma_dfs[[ds]] <- read_tsv(paste0(dirs$data_dir, "/preprocessed_GEO_data/", ds, ".tsv"))
    clinical_dfs[[ds]] <- read_tsv(paste0(dirs$data_dir, "/preprocessed_GEO_clinical_data/", ds, ".tsv"))
}

# Handle making data uniform
clinical_dfs$GSE4888 <- clinical_dfs$GSE4888 %>%
    filter(condition == "normal") %>%
    rename(sample_name = geo_accession)
rma_dfs$GSE4888 <- rma_dfs$GSE4888 %>%
    select(one_of(c("symbol", clinical_dfs$GSE4888$sample_name)))

clinical_dfs$GSE6364 <- clinical_dfs$GSE6364 %>%
    rename(sample_name = geo_accession)

clinical_dfs$GSE51981 <- clinical_dfs$GSE51981 %>%
    filter(!(tissue == "unknown" | phase == "ambiguous_unknown" | condition == "unspecified_pathology")) %>%
    mutate(title = as.character(title)) %>%
    rename(sample_name = geo_accession)
rma_dfs$GSE51981 <- rma_dfs$GSE51981 %>%
    select(one_of(c("symbol", clinical_dfs$GSE51981$sample_name)))

all_clinical_df <- bind_rows(clinical_dfs$GSE4888, clinical_dfs$GSE6364, clinical_dfs$GSE51981) %>%
    rowwise() %>%
    mutate(batch = as.double(batches[[series]])) %>%
    ungroup() %>%
    mutate(
        condition = factor(condition, levels = c("normal", "endometriosis")),
        phase = factor(phase, levels = c("proliferative", "secretory_early", "secretory_mid", "secretory_late"))
    )


Rows: 1068 Columns: 11

── Column specification ────────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: "\t"
chr (9): Division, Category, Gene Symbol, Gene Name, Synonyms, UniProt_IDs, ...
dbl (2): HGNC_IDs, HGNC_IDs Links


ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.

Rows: 21407 Columns: 28

── Column specification ────────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: "\t"
chr  (1): symbol
dbl (27): GSM109814, GSM109815, GSM109816, GSM109817, GSM109818, GSM109819, ...


ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.

Rows: 27 Columns: 7

── Column specification ────────────────────────────────────────────────────────────────────────────────────────────────
Delimiter:

In [5]:
# Only need the "symbol" column from the first matrix (since all have same genes)
all_rma_df <- bind_cols(rma_dfs$GSE4888, rma_dfs$GSE6364[-1], rma_dfs$GSE51981[-1])

In [6]:
all_rma_df

symbol,GSM109815,GSM109816,GSM109817,GSM109828,GSM109829,GSM109830,GSM109831,GSM150190,GSM150191,...,GSM1256786,GSM1256787,GSM1256788,GSM1256791,GSM1256793,GSM1256796,GSM1256797,GSM1256798,GSM1256799,GSM1256800
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,...,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
A1BG,5.685566,6.309144,6.558803,5.676761,5.713494,5.651049,6.095096,5.854344,5.447841,...,6.145047,6.126593,6.599092,5.970239,6.057815,6.441339,5.991864,6.485949,6.274268,6.409687
A1BG-AS1,6.354643,7.025457,7.019929,6.820971,6.986295,7.019873,6.614798,6.777897,7.479844,...,5.856890,5.941728,5.879944,6.097081,5.844443,5.913063,6.025716,5.814924,5.692197,5.790890
A1CF,6.017202,6.806242,5.876322,6.075240,6.048896,6.123009,6.480756,6.264393,6.434380,...,5.837212,5.564375,5.263638,5.859302,5.610516,5.896221,5.446063,5.874074,5.643722,5.432071
A2M,10.705147,10.982733,11.381476,11.434625,11.319421,11.669190,11.679939,10.869828,10.388754,...,10.761609,10.889744,11.732717,11.766133,11.044853,11.807358,11.383761,10.991057,11.188148,10.986126
A2M-AS1,6.193356,6.496668,6.377511,6.072465,5.664001,6.158695,5.950787,6.412899,5.860931,...,6.304126,6.635071,7.063681,5.283829,6.359341,6.337862,6.054985,6.940392,6.448001,5.843957
A2ML1,5.936272,6.090145,5.661428,5.772800,5.649532,5.542943,6.006635,5.731372,6.354088,...,4.545159,4.468034,4.856192,4.792261,4.448566,4.545780,4.483322,4.300889,4.633475,4.386206
A2MP1,6.193356,6.532157,5.972719,6.051718,6.131620,6.140529,6.239424,6.152233,6.558805,...,5.449048,5.649246,5.601276,5.735641,5.779447,5.851594,5.304028,5.725718,5.835786,5.564395
A4GALT,7.399494,7.314846,7.260542,6.702430,7.204924,7.250284,7.084704,6.865608,7.216150,...,6.635044,6.238798,5.937796,6.593342,6.728298,6.532459,6.351160,6.270813,6.634007,6.556477
A4GNT,6.212988,6.594580,6.254428,6.024918,6.118253,6.123767,6.236670,6.122098,6.744604,...,5.807315,5.734504,5.434892,5.637010,5.550587,5.607756,5.769874,5.554944,5.787562,5.690706


In [10]:
all_clinical_df

sample_name,series,title,condition,endometriosis_stage,phase,tissue,batch
<chr>,<chr>,<chr>,<fct>,<chr>,<fct>,<chr>,<dbl>
GSM109815,GSE4888,endometrium M165,normal,none,proliferative,uterus,1
GSM109816,GSE4888,endometrium M169,normal,none,proliferative,uterus,1
GSM109817,GSE4888,endometrium M182,normal,none,proliferative,uterus,1
GSM109828,GSE4888,endometrium G98A,normal,none,secretory_mid,uterus,1
GSM109829,GSE4888,endometrium M153,normal,none,secretory_mid,uterus,1
GSM109830,GSE4888,endometrium M158,normal,none,secretory_mid,uterus,1
GSM109831,GSE4888,endometrium M163,normal,none,secretory_mid,uterus,1
GSM150190,GSE6364,PE_D_26A,endometriosis,moderate_severe,proliferative,uterus,2
GSM150191,GSE6364,PE_D_508,endometriosis,moderate_severe,proliferative,uterus,2


In [15]:
all(colnames(all_rma_df)[-1] == all_clinical_df$sample_name)

[1] TRUE

# Prep data

In [16]:
rma_expr <- as.matrix(all_rma_df[-1])
rownames(rma_expr) <- all_rma_df$symbol
model_m <- model.matrix(~ condition, data = all_clinical_df)
batch <- all_clinical_df$batch

# Batch Correction

## Choose reference batch

In [20]:
ref_batch <- match("GSE4888", dsets)

In [21]:
ref_batch

[1] 1

## THE MAGIC

In [27]:
bc_rma_expr <- ComBat(rma_expr, batch = batch, mod = model_m, BPPARAM = mc_param, ref.batch = ref_batch)

Using batch =1as a reference batch (this batch won't change)

Found3batches

Adjusting for1covariate(s) or covariate level(s)

Standardizing Data across genes

Fitting L/S model and finding priors

Finding parametric adjustments

Adjusting the Data




In [28]:
bc_rma_expr_df <- bc_rma_expr %>%
    as_tibble(rownames = "symbol")

In [29]:
bc_rma_expr_df

symbol,GSM109815,GSM109816,GSM109817,GSM109828,GSM109829,GSM109830,GSM109831,GSM150190,GSM150191,...,GSM1256786,GSM1256787,GSM1256788,GSM1256791,GSM1256793,GSM1256796,GSM1256797,GSM1256798,GSM1256799,GSM1256800
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,...,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
A1BG,5.685566,6.309144,6.558803,5.676761,5.713494,5.651049,6.095096,6.734897,6.311050,...,5.807815,5.799219,6.019293,5.726396,5.767185,5.945817,5.736468,5.966595,5.868001,5.931074
A1BG-AS1,6.354643,7.025457,7.019929,6.820971,6.986295,7.019873,6.614798,7.162332,7.692469,...,6.700733,6.763380,6.717756,6.878098,6.691541,6.742213,6.825399,6.669743,6.579118,6.651996
A1CF,6.017202,6.806242,5.876322,6.075240,6.048896,6.123009,6.480756,6.343271,6.492033,...,6.245288,6.031817,5.796515,6.262572,6.067918,6.291459,5.939247,6.274130,6.093899,5.928299
A2M,10.705147,10.982733,11.381476,11.434625,11.319421,11.669190,11.679939,11.021458,10.674658,...,11.147112,11.202096,11.563820,11.578159,11.268654,11.595849,11.414081,11.245570,11.330142,11.243454
A2M-AS1,6.193356,6.496668,6.377511,6.072465,5.664001,6.158695,5.950787,6.176829,5.792399,...,6.171834,6.317519,6.506197,5.722692,6.196140,6.186685,6.062160,6.451924,6.235170,5.969264
A2ML1,5.936272,6.090145,5.661428,5.772800,5.649532,5.542943,6.006635,5.752719,6.293694,...,5.671488,5.636839,5.811221,5.782499,5.628093,5.671767,5.643708,5.561749,5.711164,5.600078
A2MP1,6.193356,6.532157,5.972719,6.051718,6.131620,6.140529,6.239424,6.046551,6.383859,...,6.038329,6.151578,6.124441,6.200450,6.225230,6.266043,5.956293,6.194836,6.257100,6.103579
A4GALT,7.399494,7.314846,7.260542,6.702430,7.204924,7.250284,7.084704,7.514505,7.745656,...,7.101046,6.920610,6.783544,7.082056,7.143511,7.054332,6.971775,6.935188,7.100574,7.065269
A4GNT,6.212988,6.594580,6.254428,6.024918,6.118253,6.123767,6.236670,6.186598,6.752778,...,6.290891,6.244271,6.052436,6.181848,6.126513,6.163117,6.266918,6.129302,6.278243,6.216228
